# 분석 준비

In [43]:
import pandas as pd

In [ ]:
df = pd.read_csv('D:/마이데이터/MEASURED_WAVE_TRAIN.csv', encoding = 'cp949')
data = pd.read_csv('D:/마이데이터/REFERENCE_WAVE.csv', encoding = 'cp949')
target = pd.read_csv('D:/마이데이터/MEASURED_WAVE_TEST.csv', encoding = 'cp949')

In [ ]:
for i in tqdm(range(df.shape[0])):
    a = df['MEASURED_WAVE'][i].split(',')
    for x in range(len(a)):
        df['{}'.format(x)][i] = a[x]

In [ ]:
for i in range(200):
    data['{}'.format(i)] = 0

In [ ]:
for i in tqdm(range(data.shape[0])):
    a = data['REFERENCE_WAVE'][i].split(',')
    for x in range(len(a)):
        data['{}'.format(x)][i] = a[x]

In [ ]:
for i in range(200):
    target['{}'.format(i)] = 0

for i in tqdm(range(target.shape[0])):
    a = target['MEASURED_WAVE'][i].split(',')
    for x in range(len(a)):
        target['{}'.format(x)][i] = a[x]

In [ ]:
# df.to_csv('D:/마이데이터/MEASURED_WAVE_TRAIN_처리.csv')
# data.to_csv('D:/마이데이터/REFERENCE_WAVE_처리.csv')
# target.to_csv('D:/마이데이터/MEASURED_WAVE_TEST_처리.csv')

# 분석시작

In [54]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from numpy.random import seed
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm
from scipy import stats
import joblib
from sklearn.manifold import TSNE
from lightgbm import LGBMClassifier
from sklearn.metrics import confusion_matrix, accuracy_score
from sklearn.metrics import precision_score, recall_score
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.metrics import classification_report

In [55]:
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow.keras import optimizers, Sequential
from tensorflow.keras.models import Model
from keras.utils.vis_utils import plot_model
from tensorflow.keras.layers import Dense, LSTM, RepeatVector, TimeDistributed
from tensorflow.keras.callbacks import ModelCheckpoint, TensorBoard
from tensorflow.keras import regularizers



from numpy.random import seed
from sklearn.model_selection import train_test_split
from datetime import datetime
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm
from scipy import stats

In [56]:
# 일반 데이터
df = pd.read_csv('D:/마이데이터/MEASURED_WAVE_TRAIN_처리.csv', encoding = 'cp949')
df = df.drop(columns = ['Unnamed: 0','MEASURED_WAVE'])
df[df.iloc[:,3:].columns] = df.iloc[:,3:].astype('float')
df['QUALITY'] = df['QUALITY'].apply(lambda x: 1 if x == 'BAD' else 0)
ee = df.iloc[:,2:]
df['max'] = ee.T.max()
df['min'] = ee.T.min()
df['q'] = stats.kurtosis(ee.T)
df['qq'] = stats.skew(ee.T)
print('일반데이터 길이:', len(df))
qq = df
data = pd.read_csv('D:/마이데이터/REFERENCE_WAVE_처리.csv', encoding = 'cp949')
data = data.drop(columns = ['Unnamed: 0','REFERENCE_WAVE'])
data[data.iloc[:,3:].columns] = data.iloc[:,3:].astype('float')
print('확실한 데이터 길이:', len(data))
ee = data.iloc[:,2:]
data['max'] = ee.T.max()
data['min'] = ee.T.min()
data['q'] = stats.kurtosis(ee.T)
data['qq'] = stats.skew(ee.T)
# df = df.iloc[:,2:]
# data = data.iloc[:,2:]
qqq = data
df = pd.concat([df,data]).reset_index().drop(columns = 'index')
df['QUALITY'] = df['QUALITY'].fillna(10)

sub = pd.read_csv('D:/마이데이터/MEASURED_WAVE_TEST_처리.csv', encoding = 'cp949')
sub = sub.drop(columns = ['Unnamed: 0','INDEX','MEASURED_WAVE'])
ee = sub.iloc[:,2:]
sub['max'] = ee.T.max()
sub['min'] = ee.T.min()
sub['q'] = stats.kurtosis(ee.T)
sub['qq'] = stats.skew(ee.T)
print('답안지 데이터 길이:', len(sub))

df = pd.concat([df,sub]).reset_index().drop(columns = 'index')

x = pd.concat([qq,sub]).reset_index().drop(columns = 'index')
xt = x['QUALITY']
xx = df.copy()
xxx = df.copy()
REFERENCE_ID = x['REFERENCE_ID']

일반데이터 길이: 27176
확실한 데이터 길이: 9590
답안지 데이터 길이: 5354


In [57]:
x

,REFERENCE_ID,QUALITY,0,1,2,3,4,5,6,7,...,194,195,196,197,198,199,max,min,q,qq
0,faea7610122d0747,0.0,556,564.0,577.0,587.0,596.0,606.0,617.0,627.0,...,608.0,582.0,553.0,527.0,504.0,479.0,3530.0,479.0,-1.454166,-0.169549
1,9f3578aa0a830cf4,0.0,510,519.0,527.0,536.0,549.0,559.0,571.0,595.0,...,482.0,461.0,438.0,418.0,399.0,379.0,3876.0,379.0,-1.468737,-0.125712
2,78e85aaa12ee4baf,0.0,625,637.0,662.0,670.0,676.0,688.0,738.0,766.0,...,535.0,483.0,456.0,430.0,407.0,390.0,3472.0,390.0,-1.404306,-0.246982
3,a92e988962a02594,0.0,553,565.0,574.0,601.0,616.0,629.0,644.0,676.0,...,501.0,476.0,453.0,411.0,392.0,374.0,3920.0,374.0,-1.426451,-0.243389
4,c963fce20f8d8da0,0.0,398,399.0,411.0,415.0,420.0,432.0,437.0,442.0,...,322.0,308.0,296.0,270.0,256.0,243.0,3605.0,243.0,-1.542116,0.036258
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32525,15e9984a4694cacc,NaN,616,622.0,642.0,655.0,668.0,694.0,707.0,724.0,...,436.0,419.0,401.0,385.0,352.0,337.0,3943.0,337.0,-1.392741,-0.214638
32526,0a9e57e09abc6b76,NaN,666,677.0,687.0,701.0,711.0,726.0,739.0,770.0,...,514.0,495.0,470.0,448.0,427.0,405.0,3514.0,405.0,-1.377915,-0.266819
32527,c6566bde6e6ea9e1,NaN,503,506.0,521.0,528.0,536.0,550.0,559.0,569.0,...,529.0,502.0,479.0,456.0,412.0,400.0,4172.0,400.0,-1.477051,-0.165971
32528,620df0d0b216bca4,NaN,323,335.0,342.0,358.0,364.0,372.0,386.0,395.0,...,465.0,440.0,398.0,379.0,359.0,320.0,4676.0,320.0,-1.509371,0.056324


In [58]:
x['idxmax'] = x.iloc[:,2:-4].T.idxmax().astype('int')
x['idxmin'] = x.iloc[:,2:-5].T.idxmin().astype('int')

In [59]:
qqq['idxmax'] = qqq.iloc[:,1:-4].T.idxmax().astype('int')
qqq['idxmin'] = qqq.iloc[:,1:-5].T.idxmin().astype('int')

In [60]:
xq = x.drop(columns = ['QUALITY'])
wqq = xq.reset_index().set_index('REFERENCE_ID')
wqqq = qqq.set_index('REFERENCE_ID')
wqqq['index'] = 0
qqww = wqq[['index','max','min','idxmax','idxmin']] - wqqq[['index','max','min','idxmax','idxmin']]
dd = qqww.reset_index().set_index('index').sort_index().reset_index().drop(columns = 'index')

In [61]:
x['idxmax'] = dd['idxmax']
x['idxmin'] = dd['idxmin']
x['max'] = dd['max']
x['min'] = dd['min']

In [62]:
x

,REFERENCE_ID,QUALITY,0,1,2,3,4,5,6,7,...,196,197,198,199,max,min,q,qq,idxmax,idxmin
0,faea7610122d0747,0.0,556,564.0,577.0,587.0,596.0,606.0,617.0,627.0,...,553.0,527.0,504.0,479.0,10.0,37.0,-1.454166,-0.169549,9,0
1,9f3578aa0a830cf4,0.0,510,519.0,527.0,536.0,549.0,559.0,571.0,595.0,...,438.0,418.0,399.0,379.0,416.0,-30.0,-1.468737,-0.125712,-1,0
2,78e85aaa12ee4baf,0.0,625,637.0,662.0,670.0,676.0,688.0,738.0,766.0,...,456.0,430.0,407.0,390.0,-7.0,-22.0,-1.404306,-0.246982,0,0
3,a92e988962a02594,0.0,553,565.0,574.0,601.0,616.0,629.0,644.0,676.0,...,453.0,411.0,392.0,374.0,-228.0,20.0,-1.426451,-0.243389,5,0
4,c963fce20f8d8da0,0.0,398,399.0,411.0,415.0,420.0,432.0,437.0,442.0,...,296.0,270.0,256.0,243.0,177.0,-22.0,-1.542116,0.036258,-4,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32525,15e9984a4694cacc,NaN,616,622.0,642.0,655.0,668.0,694.0,707.0,724.0,...,401.0,385.0,352.0,337.0,-245.0,-53.0,-1.392741,-0.214638,-24,0
32526,0a9e57e09abc6b76,NaN,666,677.0,687.0,701.0,711.0,726.0,739.0,770.0,...,470.0,448.0,427.0,405.0,-4.0,-7.0,-1.377915,-0.266819,-1,0
32527,c6566bde6e6ea9e1,NaN,503,506.0,521.0,528.0,536.0,550.0,559.0,569.0,...,479.0,456.0,412.0,400.0,43.0,0.0,-1.477051,-0.165971,-1,0
32528,620df0d0b216bca4,NaN,323,335.0,342.0,358.0,364.0,372.0,386.0,395.0,...,398.0,379.0,359.0,320.0,519.0,45.0,-1.509371,0.056324,11,0


In [63]:
z = x.copy()

In [64]:
x = x.drop(columns = 'QUALITY')

In [65]:
cc = x.iloc[:,1:-6]
ccc = qqq.iloc[:,1:-6]

In [66]:
d = ((cc.T - cc.T.shift(1))/cc.T).T.drop(columns = '0')
dd = ((ccc.T - ccc.T.shift(1))/ccc.T).T.drop(columns = '0')

In [67]:
d.columns = range(601,800)
dd.columns = range(601,800)

In [68]:
xr = x['REFERENCE_ID']
qr = qqq['REFERENCE_ID']

x = x.drop(columns = 'REFERENCE_ID')
qqq = qqq.drop(columns = 'REFERENCE_ID')

In [69]:
x.columns = range(206)

In [70]:
qqq.columns = range(206)

In [71]:
x['REFERENCE_ID'] = xr
qqq['REFERENCE_ID'] = qr

In [72]:
s = x

In [73]:
ss = qqq

In [74]:
s[d.columns] = d
ss[dd.columns] = dd

In [75]:
wqq = s.reset_index().set_index('REFERENCE_ID')
wqqq = ss.set_index('REFERENCE_ID')
wqqq['index'] = 0
qqww = wqq - wqqq
x = qqww.reset_index().set_index('index').sort_index().reset_index().drop(columns = 'index')
we = x.iloc[:,1:]
we.columns = we.columns.astype('int')
we = we.T.sort_index().T
we['REFERENCE_ID'] = x['REFERENCE_ID']
x = we.copy()

In [76]:
x['target'] = xt

In [77]:
x

,0,1,2,3,4,5,6,7,8,9,...,792,793,794,795,796,797,798,799,REFERENCE_ID,target
0,-24.0,-26.0,-27.0,-29.0,-31.0,-33.0,-34.0,-36.0,-39.0,-41.0,...,0.000495,-0.001178,-0.001159,0.000186,-0.001361,0.000149,-0.002624,-0.000156,faea7610122d0747,0.0
1,-55.0,-54.0,-54.0,-54.0,-54.0,-53.0,-53.0,-54.0,-54.0,-53.0,...,-0.002730,-0.002645,-0.002245,-0.002870,-0.003471,-0.000972,-0.003330,-0.003871,9f3578aa0a830cf4,0.0
2,-4.0,-5.0,4.0,1.0,-7.0,-11.0,10.0,18.0,13.0,6.0,...,0.002406,-0.006647,-0.003251,-0.005864,-0.008896,-0.012113,-0.005703,0.007381,78e85aaa12ee4baf,0.0
3,17.0,17.0,16.0,16.0,16.0,15.0,15.0,15.0,14.0,14.0,...,0.000408,0.000773,0.002411,0.000576,0.000390,0.000374,-0.000082,0.002719,a92e988962a02594,0.0
4,-7.0,-7.0,-6.0,-5.0,-4.0,-4.0,-3.0,-2.0,-1.0,0.0,...,-0.001330,-0.001466,0.000124,-0.000678,-0.000168,-0.001058,-0.000924,-0.000668,c963fce20f8d8da0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32525,108.0,110.0,115.0,119.0,122.0,128.0,132.0,136.0,140.0,148.0,...,0.006920,0.006049,0.005789,0.005798,0.005960,0.005005,0.006250,0.006772,15e9984a4694cacc,NaN
32526,24.0,20.0,17.0,18.0,16.0,16.0,14.0,13.0,11.0,11.0,...,-0.004258,-0.006533,0.008020,0.013930,-0.004668,-0.002750,-0.002991,-0.003350,0a9e57e09abc6b76,NaN
32527,8.0,8.0,6.0,6.0,4.0,0.0,-3.0,-7.0,-9.0,-7.0,...,0.000760,-0.022728,0.005671,-0.004182,0.001983,0.002193,-0.015887,0.015000,c6566bde6e6ea9e1,NaN
32528,-66.0,-71.0,-74.0,-80.0,-83.0,-86.0,-92.0,-96.0,-102.0,-106.0,...,-0.003514,-0.003052,-0.004241,-0.003342,-0.002283,-0.003836,-0.003762,-0.001875,620df0d0b216bca4,NaN


In [78]:
sub = x[x['target'].isna()].reset_index().drop(columns = 'index')
x = x.dropna()

In [79]:
target = x['target']
x = x.drop(columns = 'target')
sub = sub.drop(columns = 'target')

In [80]:
x = x.drop(columns = 'REFERENCE_ID')
sub = sub.drop(columns = 'REFERENCE_ID')

In [81]:
x

,0,1,2,3,4,5,6,7,8,9,...,790,791,792,793,794,795,796,797,798,799
0,-24.0,-26.0,-27.0,-29.0,-31.0,-33.0,-34.0,-36.0,-39.0,-41.0,...,-0.001609,-0.000562,0.000495,-0.001178,-0.001159,0.000186,-0.001361,0.000149,-0.002624,-0.000156
1,-55.0,-54.0,-54.0,-54.0,-54.0,-53.0,-53.0,-54.0,-54.0,-53.0,...,-0.004174,-0.002489,-0.002730,-0.002645,-0.002245,-0.002870,-0.003471,-0.000972,-0.003330,-0.003871
2,-4.0,-5.0,4.0,1.0,-7.0,-11.0,10.0,18.0,13.0,6.0,...,-0.008426,-0.007641,0.002406,-0.006647,-0.003251,-0.005864,-0.008896,-0.012113,-0.005703,0.007381
3,17.0,17.0,16.0,16.0,16.0,15.0,15.0,15.0,14.0,14.0,...,0.002081,0.000536,0.000408,0.000773,0.002411,0.000576,0.000390,0.000374,-0.000082,0.002719
4,-7.0,-7.0,-6.0,-5.0,-4.0,-4.0,-3.0,-2.0,-1.0,0.0,...,0.000356,-0.000864,-0.001330,-0.001466,0.000124,-0.000678,-0.000168,-0.001058,-0.000924,-0.000668
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27171,85.0,100.0,109.0,131.0,139.0,146.0,156.0,167.0,191.0,206.0,...,-0.010417,-0.001063,0.009541,0.000147,-0.002922,0.029694,0.025175,0.019364,0.021547,-0.008701
27172,113.0,114.0,114.0,117.0,118.0,118.0,119.0,122.0,123.0,124.0,...,0.005299,0.004055,0.006200,0.003996,0.004992,0.004762,0.005891,0.003563,0.006373,0.004649
27173,90.0,91.0,93.0,84.0,86.0,87.0,89.0,86.0,95.0,97.0,...,-0.006694,0.009979,-0.003978,0.020808,0.014240,-0.002934,-0.010600,0.027600,-0.007533,0.034580
27174,-79.0,-80.0,-81.0,-82.0,-84.0,-85.0,-86.0,-87.0,-90.0,-91.0,...,-0.004133,-0.003968,-0.003171,-0.003527,-0.003918,-0.001854,-0.004914,-0.002683,-0.002964,-0.003968


In [82]:
#StandaerScaler
sclaer = StandardScaler()
sclaer.fit(x)
xx = sclaer.transform(x)
subsub = sclaer.transform(sub)

In [83]:
xx

array([[-0.446399  , -0.46868699, -0.4731513 , ...,  0.02728112,
        -0.26638212, -0.01107707],
       [-1.03644191, -0.99036839, -0.96464612, ..., -0.09534061,
        -0.34110757, -0.37029994],
       [-0.06572616, -0.07742594,  0.09115755, ..., -1.3145492 ,
        -0.59230496,  0.71774913],
       ...,
       [ 1.7234362 ,  1.711196  ,  1.71127008, ...,  3.03127771,
        -0.78603911,  3.34785921],
       [-1.49324932, -1.47478683, -1.45614093, ..., -0.28255783,
        -0.30238645, -0.37973406],
       [ 1.01919144,  0.96593686,  0.96492611, ..., -0.3216208 ,
        -0.11592656, -0.09968967]])

In [89]:
subsub

array([[ 0.23881212,  0.20204624,  0.34600672, ..., -0.86488644,
        -0.38236854, -0.64014652],
       [ 0.42914854,  0.42562398,  0.40061725, ...,  0.2661588 ,
         0.05541228,  0.29909864],
       [-1.74068666, -1.6797331 , -1.61997253, ..., -0.68713588,
        -0.5404095 , -0.48618931],
       ...,
       [ 0.16267755,  0.16478328,  0.12756458, ...,  0.25098731,
        -1.67049571,  1.45449233],
       [-1.24581197, -1.30710353, -1.32871635, ..., -0.40871599,
        -0.38686918, -0.17731682],
       [-1.37904746, -1.45615536, -1.52895497, ..., -0.19851023,
        -0.24684815, -0.12481114]])

In [86]:
bb = pd.DataFrame(subsub)

In [88]:
bb.to_csv('D:/마이데이터/subsub.csv')

In [32]:
from sklearn.model_selection import train_test_split
from lightgbm import LGBMClassifier
from sklearn.model_selection import StratifiedKFold
from lightgbm import LGBMClassifier
from sklearn.metrics import confusion_matrix, accuracy_score
from sklearn.metrics import precision_score, recall_score
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.metrics import classification_report
from catboost import CatBoostClassifier,Pool

In [33]:
y = target
X = xx

In [34]:
lgbmm = LGBMClassifier(n_estimators = 10000, 
                           learning_rate= 0.01,
                           max_depth = -1,
                           min_data_in_leaf = 500,
                           num_leaves = 100,
                           metric = 'binary_logloss',
                       bagging_fraction = 0.3,
                           num_iterations = 1000,
                       scale_pos_weight=100,
                           random_state = 42,
                      n_jobs = -1)
lgbmm.fit(X, y)

c:\users\ku_bd_3\anaconda3\envs\bigdata\lib\site-packages\lightgbm\engine.py:148: UserWarning: Found `num_iterations` in params. Will use it instead of argument
  _log_warning("Found `{}` in params. Will use it instead of argument".format(alias))


[LightGBM] [Warning] min_data_in_leaf is set=500, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=500
[LightGBM] [Warning] bagging_fraction is set=0.3, subsample=1.0 will be ignored. Current value: bagging_fraction=0.3


LGBMClassifier(bagging_fraction=0.3, learning_rate=0.01,
               metric='binary_logloss', min_data_in_leaf=500,
               n_estimators=10000, num_iterations=1000, num_leaves=100,
               random_state=42, scale_pos_weight=100)

In [35]:
import joblib

In [36]:
joblib.dump(lgbmm, 'D:/마이데이터/lgbmclassifier.pkl')

['D:/마이데이터/lgbmclassifier.pkl']

In [37]:
ekq = pd.read_csv('D:/마이데이터/ANSWER_FORM.csv')
y_pred = (lgbmm.predict_proba(subsub)[:,1] >= 0.55).astype(bool)
ekq['QUALITY'] = y_pred
ekq['QUALITY'] = ekq['QUALITY'].apply(lambda x: 'BAD' if x == True else 'GOOD')
ekq.to_csv('D:/마이데이터/새 폴더/2.csv')

In [38]:
y_pred

array([False, False,  True, ..., False,  True,  True])

In [39]:
ekq

,INDEX,QUALITY
0,0,GOOD
1,1,GOOD
2,2,BAD
3,3,BAD
4,4,BAD
...,...,...
5349,5349,BAD
5350,5350,GOOD
5351,5351,GOOD
5352,5352,BAD


In [42]:
lgbmm.feature_importances_
fi = pd.DataFrame(x.columns)
fi['import'] = lgbmm.feature_importances_
fi.sort_values(by = 'import',ascending=False).head()

,0,import
203,왜도,2580
200,최댓값,2425
201,최솟값,1677
362,기울기 157,920
374,기울기 169,871


In [41]:
for i in range(len(fi)):
    if fi[0][i] == 203:
        fi[0][i] = '왜도'
    elif fi[0][i] == 200:
        fi[0][i] = '최댓값'
    elif fi[0][i] == 201:
        fi[0][i] = '최솟값'
    elif fi[0][i] == 757:
        fi[0][i] = '기울기 157'
    elif fi[0][i] == 769:
        fi[0][i] = '기울기 169 '
    else:
        fi[0][i] = fi[0][i]
    

c:\users\ku_bd_3\anaconda3\envs\bigdata\lib\site-packages\ipykernel_launcher.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  del sys.path[0]
c:\users\ku_bd_3\anaconda3\envs\bigdata\lib\site-packages\ipykernel_launcher.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  """
c:\users\ku_bd_3\anaconda3\envs\bigdata\lib\site-packages\ipykernel_launcher.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  import sys
c:\users\ku_bd_3\ana